\begin{align*}
p_k = P(y=k x; \theta) = \frac{e^{\theta_k^T x}}{\sum_{j=1}^K e^{\theta_j^T x}}\\
\min_{\theta} L = - \sum_{j=1}^K y_j \log(p_j)\\
L = - \sum_{j=1}^K y_j ( \theta_j^T x - \log \sum_{l=1}^K e^{\theta_l^T x} )\\
\frac{\partial L}{\partial \theta_k} = -(1\{y==k\}) x + \frac{e^{\theta_k^T x}}{\sum_{l=1}^K e^{\theta_l^T x}} x\\
\nabla_{\theta_k} J(\Theta) = \frac{1}{m} \sum_{i=1}^m (p_k^{(i)} - 1\{y^{(i)}==k\}) x^{(i)}\\
\theta_k := \theta_k - \alpha (\frac{1}{m} \sum_{i=1}^m (p_k^{(i)} - 1\{y^{(i)}==k\}) x^{(i)})\\
\theta_k := \theta_k - \alpha (\frac{1}{m}(P_k - y_k) X)
\end{align*}

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('data/iris.csv')

In [3]:
X,y=df.loc[:,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:,'Species']

In [4]:
y=y.map({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2})

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [6]:
X_train_std=X_train.std(axis=0)
X_train_mean=X_train.mean(axis=0)
X_train=(X_train-X_train_mean)/X_train_std
X_test=(X_test-X_train_mean)/X_train_std

In [7]:
X_train.insert(0,'x0',np.ones(X_train.shape[0]))
X_test.insert(0,'x0',np.ones(X_test.shape[0]))

In [8]:
class SoftMax:
    def __init__(self,eps=1e-5,max_iter=100,lr=0.1,l2=0):
        self.eps=eps
        self.max_iter=max_iter
        self.lr=lr
        self.l2=l2
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        m,n=self.X.shape
        self.classes=np.unique(y)
        self.n_classes=len(self.classes)
        self.theta={c:np.zeros(n) for c in self.classes}
        
        it=0
        max_diff=2*self.eps
        while max_diff>=self.eps and it<self.max_iter:
            it+=1
            prev_theta=self.theta.copy()
            max_diff=0
            prev_theta=self.theta.copy()
            for c in self.classes:
                theta_c=self.theta[c]
                P_c=np.exp(X@theta_c)/np.sum([np.exp(X@prev_theta[j])for j in self.classes],axis=0)       
                self.theta[c]=theta_c-self.lr*((P_c-(y==c).astype('int32'))@self.X/m+self.l2*self.theta[c])
                diff=np.linalg.norm(self.theta[c]-prev_theta[c],ord=1)
                if diff>max_diff:
                    max_diff=diff
            
            #print(max_diff)
        print(f"total iterations: {it}")
    def predict(self,X):
        m,n=X.shape
        probs=np.zeros((m,self.n_classes))
        for i,c in enumerate(self.classes):
            theta_c=self.theta[c]    
            P_c=np.exp(X@theta_c)/np.sum([np.exp(X@self.theta[j])for j in self.classes],axis=0)
            probs[:,i]=P_c
        results=np.argmax(probs,axis=1)
        return self.classes[results]

In [9]:
SM=SoftMax(eps=1e-5,max_iter=100000,lr=1,l2=1e-4)
SM.fit(X_train.values,y_train.values)
preds=SM.predict(X_test.values)
print(f"Accuracy: {np.mean(preds==y_test)*100}%")

total iterations: 21007
Accuracy: 100.0%
